In [2]:
import os
import sys
module_path = os.path.abspath(os.path.join(r'C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\src'))

if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
from dask_cluster_manager import DaskClusterManager


ee_key_host_path =r'C:\Users\Adriano Matos\Documents\credentials\robust-raster-cefaedc5282c.json' # Path in the host machine
ee_key_container_path = '/credentials/robust-raster-cefaedc5282c.json'  # Path inside the container

volumes = {
    ee_key_host_path: {'bind': ee_key_container_path, 'mode': 'ro'},  # 'ro' means read-only
}
dcm = DaskClusterManager()
dcm.create_test_cluster(volumes=volumes)

Dask Scheduler started with ID 4f6256c5e3e4e5d31cb4f1aa842c7758bd6842a1cb2f39a657d5b520829ba7f4
Dask Worker started with ID 59d28d919780aa4e35223725f6fc2b839cae58e939d46993fe003a6fbeb36921
Test cluster created with 1 worker and half the system memory.
Dask dashboard available at http://localhost:8787


In [1]:
dcm.stop_and_remove_containers()

NameError: name 'dcm' is not defined

In [5]:
dask_client = dcm.get_dask_client()

Connected to Dask Scheduler.


In [4]:
logs = scheduler.logs()
print(logs.decode("utf-8"))

/usr/local/lib/python3.10/site-packages/distributed/cli/dask_scheduler.py:142: FutureWarning: dask-scheduler is deprecated and will be removed in a future release; use `dask scheduler` instead
  warnings.warn(
2024-09-19 20:48:31,141 - distributed.scheduler - INFO - -----------------------------------------------
2024-09-19 20:48:31,569 - distributed.http.proxy - INFO - To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2024-09-19 20:48:31,608 - distributed.scheduler - INFO - State start
2024-09-19 20:48:31,611 - distributed.scheduler - INFO - -----------------------------------------------
2024-09-19 20:48:31,612 - distributed.scheduler - INFO -   Scheduler at:     tcp://172.20.0.2:8786
2024-09-19 20:48:31,612 - distributed.scheduler - INFO -   dashboard at:  http://172.20.0.2:8787/status
2024-09-19 20:48:31,612 - distributed.scheduler - INFO - Registering Worker plugin shuffle
2024-09-19 20:48:32,699 - distribute

In [6]:
from dask_plugins import EEPlugin

ee_plugin = EEPlugin(ee_key_container_path)
dask_client.register_plugin(ee_plugin)

{'tcp://172.20.0.3:32835': {'status': 'OK'}}

In [13]:
# EE credentials with a JSON key
import json
import ee
json_key = r'C:\Users\Adriano Matos\Documents\credentials\robust-raster-cefaedc5282c.json'
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)

In [7]:
# Earth Engine xarray
import sys
import os
import ee
import json

json_key = r'C:\Users\Adriano Matos\Documents\credentials\robust-raster-cefaedc5282c.json'
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)

ee.Initialize(credentials = credentials, opt_url='https://earthengine-highvolume.googleapis.com')

def prep_sr_l8(image):
    # Develop masks for unwanted pixels (fill, cloud, cloud shadow).
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    def get_factor_img(factor_names):
        factor_list = image.toDictionary().select(factor_names).values()
        return ee.Image.constant(factor_list)
    
    scale_img = get_factor_img([
        'REFLECTANCE_MULT_BAND_.|TEMPERATURE_MULT_BAND_ST_B10'])
    offset_img = get_factor_img([
        'REFLECTANCE_ADD_BAND_.|TEMPERATURE_ADD_BAND_ST_B10'])
    scaled = image.select('SR_B.|ST_B10').multiply(scale_img).add(offset_img)

    # Replace original bands with scaled bands and apply masks.
    return image.addBands(scaled, None, True)\
        .updateMask(qa_mask).updateMask(saturation_mask)

if module_path not in sys.path:
    sys.path.append(module_path)

from input_driver import EarthEngineData

CALIFORNIA = ee.FeatureCollection("projects/calfuels/assets/Boundaries/California")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU_remove_NV_remove_lake")

parameters = {
            'collection': 'LANDSAT/LC08/C02/T1_L2',
            'bands': ['SR_B4', 'SR_B5'],
            'start_date': '2020-12-29',
            'end_date': '2020-12-31',
            'geometry': CALIFORNIA.geometry(),
            'crs': 'EPSG:3310',
            'scale': 100,
            'map_function': prep_sr_l8
        }

ds = EarthEngineData(parameters).dataset

In [8]:
print(ds)

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 3, X: 9084, Y: 10578)
Coordinates:
  * time     (time) datetime64[ns] 24B 2020-12-29T18:57:32.281000 ... 2020-12...
  * X        (X) float64 73kB -4.216e+05 -4.215e+05 ... 4.866e+05 4.867e+05
  * Y        (Y) float64 85kB -5.992e+05 -5.991e+05 ... 4.584e+05 4.585e+05
Data variables:
    SR_B4    (time, X, Y) float32 1GB dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
    SR_B5    (time, X, Y) float32 1GB dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
Attributes: (12/26)
    date_range:             [1365638400000, 1654560000000]
    description:            <p>This dataset contains atmospherically correcte...
    keywords:               ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'l...
    period:                 0
    product_tags:           ['global', 'sr', 'reflectance', 'l8sr', 'lst', 'c...
    provider:               USGS
    ...                     ...
    visualization_1_name:   Near Infrared (543)
    visualization_2_ba

In [1]:
# Template xarray based on Earth Engine
import numpy as np
import pandas as pd
import xarray as xr

# Define the dimensions
time = pd.date_range("2020-12-29T18:57:32.281000", periods=3)
X = np.linspace(-421600, 486700, 9084)
Y = np.linspace(-599200, 458500, 10578)

# Create a data array with random data for each variable
data = np.random.rand(len(time), len(X), len(Y)).astype(np.float32)

# Create a dictionary of data variables
data_vars = {
    #'SR_B1': (['time', 'X', 'Y'], data),
    #'SR_B2': (['time', 'X', 'Y'], data),
    #'SR_B3': (['time', 'X', 'Y'], data),
    'SR_B4': (['time', 'X', 'Y'], data),
    'SR_B5': (['time', 'X', 'Y'], data),
    #'SR_B6': (['time', 'X', 'Y'], data),
    #'ST_EMSD': (['time', 'X', 'Y'], data),
    #'ST_QA': (['time', 'X', 'Y'], data),
    #'ST_TRAD': (['time', 'X', 'Y'], data),
    #'ST_URAD': (['time', 'X', 'Y'], data),
    #'QA_PIXEL': (['time', 'X', 'Y'], data),
    #'QA_RADSAT': (['time', 'X', 'Y'], data),
}

chunk_size = {'time': 3, 'X': 512, 'Y': 256}

# Create the dataset
ds = xr.Dataset(
    data_vars=data_vars,
    coords={'time': time, 'X': X, 'Y': Y},
    attrs={
        'date_range': '[1365638400000, 1654560000000]',
        'description': '<p>This dataset contains atmospherically corrected data.</p>',
        'keywords': ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'landsat'],
        'period': 0,
        'visualization_2_max': 30000.0,
        'visualization_2_min': 0.0,
        'visualization_2_name': 'Shortwave Infrared (753)',
        'crs': 'EPSG:3310'
    }
).chunk(chunk_size)


In [7]:
# Load the data using Dask Delayed!
import xarray as xr
import dask

chunk_size  = {
            'time': 3,
            'X': 512,
            'Y': 256
}
# Define a delayed function that opens the dataset
@dask.delayed
def load_dataset_on_worker():
    return xr.open_dataset('/data/saved_on_disk.nc', chunks=chunk_size)

# Trigger the delayed execution, this will happen on the worker
delayed_ds = load_dataset_on_worker()

# Optionally, you can convert the delayed object into a Dask-backed xarray Dataset
ds = delayed_ds.compute()

# Now ds can be used for further computation on the Dask workers

In [ ]:
import xarray as xr
import dask.array as da
import dask
import psutil
import time

def log_memory_usage():
    process = psutil.Process()
    return process.memory_info().rss / 1024**2  # Memory usage in MB


def process_chunk_as_pandas(df):
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    return df

def user_function_wrapper(ds, user_func, *args, **kwargs):
    df_input = ds.to_dataframe().reset_index()
    df_output = user_func(df_input, *args, **kwargs)
    df_output = df_output.set_index(list(ds.dims))
    ds_output = df_output.to_xarray()
    return ds_output


def tune_user_function(chunk, user_func, user_func2, *args, **kwargs):

    result_chunk = user_func2(ds, user_func, *args, kwargs)
    start_time = time.time()
    result_chunk.compute()
    end_time = time.time()
        
    wall_time = end_time - start_time
    memory_usage = log_memory_usage()
    print(f"Wall time: {wall_time:.2f} seconds")
    print(f"Memory usage: {memory_usage:.2f} MB")

    # Apply the processing function to this chunk
    #processed_chunk = user_func(one_chunk)

    return result_chunk

def test_run(ds, user_func, user_func2, *args, **kwargs):
    # Dynamically determine dimension names
    dim_names = list(ds.dims.keys())
            
    # Extract a single chunk to determine the output structure using dynamic dimension names
    one_chunk_slices = {dim: slice(0, ds.chunks[dim][0]) for dim in dim_names}
    one_chunk = ds.isel(**one_chunk_slices)

    test = xr.map_blocks(tune_user_function,
                            one_chunk, 
                            args=(user_func, user_func2) + args, 
                            kwargs=kwargs)

result = test_run(ds, process_chunk_as_pandas, user_function_wrapper)

In [7]:
import xarray as xr
import pandas as pd
import sys, os

def process_chunk_as_pandas(df):
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    return df

from udf_tuner import UserDefinedFunction

user_defined_func = UserDefinedFunction()

# Apply the user-defined function
# If I load the dataset as a dask delayed, I don't need a template!
result = user_defined_func.apply_user_function(ds, process_chunk_as_pandas)

In [8]:
print(result)

<xarray.Dataset> Size: 3GB
Dimensions:  (Y: 10578, time: 3, X: 9084)
Coordinates:
  * Y        (Y) float64 85kB -5.992e+05 -5.991e+05 ... 4.584e+05 4.585e+05
  * time     (time) datetime64[ns] 24B 2020-12-29T18:57:32.281000 ... 2020-12...
  * X        (X) float64 73kB -4.216e+05 -4.215e+05 ... 4.866e+05 4.867e+05
Data variables:
    SR_B5    (time, X, Y) float32 1GB dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
    ndvi     (time, X, Y) float32 1GB dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
    SR_B4    (time, X, Y) float32 1GB dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
